load data and extract template id

In [2]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv("/Users/rohanbabbar/Documents/Final Year/cwe/Archive/juliet_cwe_dataset.csv")

print("Total samples:", len(df))
print("Unique CWEs:", df["cwe"].nunique())

# -------------------------------------------------
# Extract TEMPLATE ID from filename
# Example:
# CWE121_Stack_Based_Buffer_Overflow__char_alloca_cpy_01.c
# Template = char_alloca_cpy
# -------------------------------------------------

def extract_template(filename):
    # Split on "__" and remove trailing numbers
    parts = filename.split("__")
    if len(parts) > 1:
        template = re.sub(r"_\d+.*", "", parts[1])
        return template
    return "unknown"

df["template_id"] = df["filename"].apply(extract_template)

print("Unique templates:", df["template_id"].nunique())

Total samples: 179839
Unique CWEs: 118
Unique templates: 1327


template level split

In [3]:
# -----------------------------------------------
# TEMPLATE-LEVEL SPLIT
# -----------------------------------------------

# Get unique templates

templates = df["template_id"].unique()

# Split templates (NOT samples)
train_templates, test_templates = train_test_split(
    templates,
    test_size=0.2,
    random_state=42
)

# Assign samples based on template membership
train_df = df[df["template_id"].isin(train_templates)]
test_df  = df[df["template_id"].isin(test_templates)]

print("Train samples:", len(train_df))
print("Test samples:", len(test_df))

print("Train CWEs:", train_df["cwe"].nunique())
print("Test CWEs:", test_df["cwe"].nunique())

# Sanity check: no template leakage
assert set(train_df["template_id"]).isdisjoint(set(test_df["template_id"]))
print("✅ No template leakage confirmed")

Train samples: 144109
Test samples: 35730
Train CWEs: 118
Test CWEs: 56
✅ No template leakage confirmed


In [4]:
train_df.to_csv("juliet_train_template_split.csv", index=False)
test_df.to_csv("juliet_test_template_split.csv", index=False)

print("💾 Saved template-level splits")

💾 Saved template-level splits


In [5]:
overlap = set(train_df["template_id"]) & set(test_df["template_id"])
print("\nTemplate overlap:", len(overlap))


Template overlap: 0
